# F4 — Week 12 Performance Review

**Objective**: Review the optimisation performance of F4 across all 10 submission rounds before deciding on a strategy for the next submission.

**Function**: F4 (4D input, 1D output, maximisation)

This notebook loads the Week 12 data, visualises convergence and input-space coverage, evaluates performance, and proposes strategy improvements. No optimisation loop is run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import math

# ── Function Configuration ──
FUNC_NUM = 4
N_DIMS = 4
N_INITIAL = 30
WEEK = 11
USE_LOG_SCALE = False
DATA_DIR = '../../data/f4/'

## Step 1 — Load Data

In [ ]:
# Load Week 12 data
inputs = np.load(f'{DATA_DIR}updated_inputs - Week {WEEK}.npy')
outputs = np.load(f'{DATA_DIR}updated_outputs - Week {WEEK}.npy')

n_total = len(outputs)
n_submissions = n_total - N_INITIAL

print(f'F{FUNC_NUM} — Week {WEEK} Data Summary')
print(f'  Input dimensions:  {N_DIMS}')
print(f'  Total samples:     {n_total}')
print(f'  Initial samples:   {N_INITIAL}')
print(f'  Submissions:       {n_submissions}')
print(f'  Input shape:       {inputs.shape}')
print(f'  Output shape:      {outputs.shape}')
print(f'  Best output:       {outputs.max():.6g}')
print(f'  Worst output:      {outputs.min():.6g}')
print()

# Display data table
print('Sample | ' + ' | '.join([f'x{j+1:d}' for j in range(N_DIMS)]) + ' | y')
print('-' * (10 + N_DIMS * 12 + 15))
for i in range(n_total):
    label = 'init' if i < N_INITIAL else f'wk{i - N_INITIAL + 3}'
    row = f'{i+1:>4d}({label:>4s}) | '
    row += ' | '.join([f'{inputs[i, j]:.6f}' for j in range(N_DIMS)])
    row += f' | {outputs[i]:.6g}'
    print(row)

## Step 2 — Convergence Plot

Running best (maximum) objective value over all samples.

In [ ]:
# Compute running best (maximisation)
running_best = np.maximum.accumulate(outputs)

fig, ax = plt.subplots(figsize=(10, 5))

# Split into initial and submission regions
x_all = np.arange(1, n_total + 1)

if USE_LOG_SCALE:
    # Clamp non-positive values to epsilon before log
    plot_vals = np.where(running_best > 0, running_best, 1e-300)
    ax.set_yscale('log')
    ax.set_ylabel('Running Best (log scale)')
else:
    plot_vals = running_best
    ax.set_ylabel('Running Best')

# Plot initial samples in blue
ax.plot(x_all[:N_INITIAL], plot_vals[:N_INITIAL], 'o-', color='tab:blue',
        label='Initial samples', markersize=5)

# Plot submissions in orange
ax.plot(x_all[N_INITIAL-1:], plot_vals[N_INITIAL-1:], 'o-', color='tab:orange',
        label='Submissions', markersize=5)

# Vertical separator
ax.axvline(x=N_INITIAL + 0.5, color='grey', linestyle='--', alpha=0.7,
           label='Initial / Submission boundary')

ax.set_xlabel('Sample Number')
ax.set_title(f'F{FUNC_NUM} — Convergence Plot (Week {WEEK})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Step 3 — 2D Pair Plots

Scatter plots of each unique pair of input dimensions showing spatial coverage. Initial samples in **blue** (unmarked), submission samples in **orange** (numbered by submission week).

In [ ]:
# Generate all unique pairs of input dimensions
pairs = list(combinations(range(N_DIMS), 2))
n_pairs = len(pairs)

if n_pairs == 0:
    print('Only 1 dimension — no pair plots to display.')
else:
    # Grid layout
    n_cols = min(n_pairs, 3) if n_pairs <= 6 else min(n_pairs, 5) if n_pairs <= 15 else 7
    n_rows = math.ceil(n_pairs / n_cols)
    fig_width = n_cols * 4
    fig_height = n_rows * 3.5

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height),
                             squeeze=False)

    for idx, (di, dj) in enumerate(pairs):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]

        # Initial samples — blue, unmarked
        ax.scatter(inputs[:N_INITIAL, di], inputs[:N_INITIAL, dj],
                   c='tab:blue', marker='o', s=40, alpha=0.7, label='Initial')

        # Submission samples — orange, numbered by week
        for k in range(N_INITIAL, n_total):
            week_num = k - N_INITIAL + 3  # Weeks start at 3
            ax.scatter(inputs[k, di], inputs[k, dj],
                       c='tab:orange', marker='o', s=40, alpha=0.7)
            ax.annotate(str(week_num), (inputs[k, di], inputs[k, dj]),
                        textcoords='offset points', xytext=(4, 4),
                        fontsize=7, color='tab:orange', fontweight='bold')

        ax.set_xlabel(f'x{di+1}')
        ax.set_ylabel(f'x{dj+1}')
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f'x{di+1} vs x{dj+1}')
        ax.grid(True, alpha=0.3)

    # Add legend to first subplot
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='tab:blue', label='Initial'),
                       Patch(facecolor='tab:orange', label='Submissions')]
    axes[0][0].legend(handles=legend_elements, loc='upper right', fontsize=7)

    # Hide empty subplots
    for idx in range(n_pairs, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    fig.suptitle(f'F{FUNC_NUM} — Input Space Coverage (Week {WEEK})', fontsize=14)
    fig.tight_layout()
    plt.show()

## Step 4 — Performance Evaluation

### Current Strategy (Week 9)

- **Surrogate**: MFGP Matérn-2.5 + LinearTruncatedFidelityKernel
- **Acquisition**: MF-qNEI q=4, 64 fantasies
- **Key hyperparameters**: noise_lb=1e-4, fidelity fixed at 1.0

### Performance Summary

In [ ]:
# Performance metrics
running_best = np.maximum.accumulate(outputs)
init_best = running_best[N_INITIAL - 1]

# Count improvements and detect stalling
improvements = 0
consec_no_improve = 0
max_consec_no_improve = 0
prev_best = init_best

for j in range(N_INITIAL, n_total):
    if running_best[j] > prev_best:
        improvements += 1
        consec_no_improve = 0
    else:
        consec_no_improve += 1
        max_consec_no_improve = max(max_consec_no_improve, consec_no_improve)
    prev_best = running_best[j]

stalling = max_consec_no_improve >= 3

print(f'Best value (initial):     {init_best:.6g}')
print(f'Best value (final):       {running_best[-1]:.6g}')
print(f'Improvements:             {improvements}/{n_submissions}')
print(f'Max consecutive no-improve: {max_consec_no_improve}')
print(f'Stalling (≥3 consec):     {stalling}')
print()

# Per-submission performance
print('Week | Output         | Best-so-far    | Improved?')
print('-' * 55)
for j in range(N_INITIAL, n_total):
    week_num = j - N_INITIAL + 3
    improved = '✓' if (j == N_INITIAL and outputs[j] > init_best) or \
               (j > N_INITIAL and running_best[j] > running_best[j-1]) else '✗'
    print(f'  {week_num:>2d} | {outputs[j]:>14.6g} | {running_best[j]:>14.6g} | {improved}')

### Evaluation

F4 has shown **early gains followed by stalling** with **2 improvements** in 10 submission rounds. However, the absolute improvement is large: best moved from -4.026 (initial) to 0.532, crossing from negative to positive territory.

Key observations:
- Only 2/10 submissions improved the best, but the first improvement was substantial (from -4.03 to positive)
- 7 consecutive non-improving submissions in the later rounds indicate severe stalling
- The MFGP architecture is designed for multi-fidelity data, but F4 only has single-fidelity observations — this is a fundamental mismatch
- The LinearTruncatedFidelityKernel adds unnecessary complexity and parameters for single-fidelity data
- The pair plots across 6 dimension pairs show whether the 4D input space is being explored effectively

**Stalling status**: YES — 7 consecutive submissions without improvement.

## Step 5 — Proposed Strategy Improvements

F4 shows early gains that have stalled (2/10 improvements, 7 consecutive non-improving). Key recommendation:

1. **Switch from MFGP to standard SFGP** — The MFGP with LinearTruncatedFidelityKernel is designed for multi-fidelity data, but F4 only has single-fidelity observations. The fidelity kernel adds unnecessary parameters and complexity without benefit. A standard SFGP with Matérn-2.5 ARD will be simpler and likely model the data better.

2. **Replace MF-qNEI with qLogNEI** — The multi-fidelity acquisition function is inappropriate for single-fidelity data. Standard qLogNEI with q=4 is the correct choice.

3. **Add output standardisation** — Apply Standardize(m=1) to handle the wide output range (from -4.03 to 0.53). This improves GP conditioning significantly.

4. **Increase noise lower bound** — With 30 initial + 10 submission = 40 samples in 4D, the GP may be overfitting. A noise_lb=1e-3 provides regularisation.

5. **Increase MLL restarts to 30+** — The 4D space with 40 observations needs thorough hyperparameter optimisation to avoid degenerate lengthscales.

**Priority**: HIGH — The MFGP/single-fidelity mismatch is a fundamental architecture issue that should be resolved.

## Step 6 — Week 12 Optimisation Run

Strategy changes from week 9:

1. **Surrogate: MFGP → SFGP** — Replace `SingleTaskMultiFidelityGP` with `SingleTaskGP` using Matérn-2.5 ARD (4D). The synthetic fidelity column (all 1.0) provides no information and adds unnecessary parameters. 7 consecutive stalling submissions indicate the model is fundamentally mismatched.
2. **Outcome transform: manual z-score → Standardize(m=1)** — BoTorch's built-in auto-standardisation replaces manual z-scoring, simplifying the pipeline and ensuring correct posterior untransformation.
3. **Noise floor: 1e-4 → 1e-3** — Relaxed to prevent overfitting on 40 samples in 4D with mixed-sign outputs.
4. **MLL restarts: 15 → ≥30** — More restarts to explore the multi-modal likelihood surface after switching to the simpler SFGP kernel structure.
5. **MC samples: 64 → 512** — Increased for lower-variance quasi-MC acquisition estimates.

In [ ]:
import torch
import copy
import warnings
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.constraints import GreaterThan

# ── Hyperparameter Configuration ──────────────────────────────────────────────
# All constants documented with week 9 → week 12 change rationale

KERNEL_NU = 2.5        # Matérn smoothness — unchanged
ARD_NUM_DIMS = 4       # One lengthscale per input dimension — was 5 (4+fidelity), now 4
NOISE_LB = 1e-3        # Noise floor — was 1e-4; relaxed to prevent overfitting on 40 mixed-sign samples
N_MLL_RESTARTS = 30    # MLL fitting restarts — was 15; increased for multi-modal likelihood after SFGP switch
MC_SAMPLES = 512       # Monte Carlo samples for qLogNEI — was 64; increased for lower-variance estimates
Q_BATCH = 4            # Acquisition batch size — unchanged from week 9
NUM_RESTARTS = 20      # L-BFGS restarts for acquisition optimisation — unchanged
RAW_SAMPLES = 2048     # Sobol seed points for acquisition — was 512; increased for 4D coverage
GRID_RES = 50          # Visualisation grid resolution (50×50)

print("=== F4 Week 12 Configuration ===")
print(f"Kernel: Matérn-{KERNEL_NU} ARD (d={ARD_NUM_DIMS})")
print(f"Noise floor: {NOISE_LB} (was 1e-4)")
print(f"MLL restarts: {N_MLL_RESTARTS} (was 15)")
print(f"Batch size q={Q_BATCH}")
print(f"Raw samples: {RAW_SAMPLES} (was 512)")
print(f"Acquisition restarts: {NUM_RESTARTS}")
print(f"MC samples: {MC_SAMPLES} (was 64)")
print(f"Grid resolution: {GRID_RES}×{GRID_RES}")
print(f"Surrogate: SingleTaskGP with Standardize(m=1) — replaces SingleTaskMultiFidelityGP")

In [ ]:
# ── Data Preparation: Tensor Conversion ───────────────────────────────────────
# Convert numpy arrays to PyTorch tensors — NO manual transform needed.
# Standardize(m=1) handles z-scoring internally; posterior auto-untransforms.
# Note: week 9 used a synthetic fidelity column — dropped for SFGP.

X_train = torch.tensor(inputs, dtype=torch.float64)               # (40, 4)
Y_train = torch.tensor(outputs, dtype=torch.float64).unsqueeze(-1) # (40, 1)

# Validate: no NaN/Inf
assert not torch.isnan(X_train).any(), "X_train contains NaN"
assert not torch.isnan(Y_train).any(), "Y_train contains NaN"
assert not torch.isinf(Y_train).any(), "Y_train contains Inf"

print("=== Data Preparation ===")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"Output range: [{outputs.min():.6f}, {outputs.max():.6f}]")
print(f"Contains negative outputs: {bool((outputs < 0).any())}")
print(f"Standardize(m=1) will handle z-scoring internally")

In [ ]:
# ── GP Fitting: Multi-restart MLL with Standardize(m=1) ───────────────────────
# Fit SFGP with Matérn-2.5 ARD on raw outputs (Standardize handles z-scoring)
# 30 MLL restarts to explore multi-modal likelihood surface in 4D

best_model = None
best_loss = float("inf")

warnings.filterwarnings("ignore", category=RuntimeWarning)

print(f"\n{'Restart':>8} {'Neg MLL':>12}")
print("-" * 22)

for restart in range(N_MLL_RESTARTS):
    torch.manual_seed(restart)
    
    # Construct GP with Matérn-2.5 ARD kernel + Standardize(m=1)
    likelihood = GaussianLikelihood(
        noise_constraint=GreaterThan(NOISE_LB)
    )
    covar_module = ScaleKernel(
        MaternKernel(nu=KERNEL_NU, ard_num_dims=ARD_NUM_DIMS)
    )
    model = SingleTaskGP(
        train_X=X_train,
        train_Y=Y_train,
        covar_module=covar_module,
        likelihood=likelihood,
        outcome_transform=Standardize(m=1),
    )
    
    # Randomise hyperparameters for this restart
    model.covar_module.base_kernel.lengthscale = torch.rand(1, ARD_NUM_DIMS, dtype=torch.float64) * 0.5 + 0.1
    model.covar_module.outputscale = torch.rand(1, dtype=torch.float64) * 2.0
    model.likelihood.noise = torch.tensor([max(NOISE_LB * 10, torch.rand(1).item() * 0.01)], dtype=torch.float64)
    
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    
    try:
        fit_gpytorch_mll(mll)
        # Compute loss using model's internal standardized targets
        model.eval()
        with torch.no_grad():
            output = model(X_train)
            loss = -mll(output, model.train_targets).item()
        
        print(f"{restart:>8d} {loss:>12.4f}")
        
        if loss < best_loss:
            best_loss = loss
            best_model = copy.deepcopy(model)
    except Exception:
        print(f"{restart:>8d} {'FAILED':>12}")
        continue

warnings.filterwarnings("default", category=RuntimeWarning)

assert best_model is not None, "All MLL restarts failed!"

# Count restarts that converged near best
converged_count = 0
for restart in range(N_MLL_RESTARTS):
    torch.manual_seed(restart)
    likelihood = GaussianLikelihood(noise_constraint=GreaterThan(NOISE_LB))
    covar_module = ScaleKernel(MaternKernel(nu=KERNEL_NU, ard_num_dims=ARD_NUM_DIMS))
    model = SingleTaskGP(train_X=X_train, train_Y=Y_train, covar_module=covar_module,
                         likelihood=likelihood, outcome_transform=Standardize(m=1))
    model.covar_module.base_kernel.lengthscale = torch.rand(1, ARD_NUM_DIMS, dtype=torch.float64) * 0.5 + 0.1
    model.covar_module.outputscale = torch.rand(1, dtype=torch.float64) * 2.0
    model.likelihood.noise = torch.tensor([max(NOISE_LB * 10, torch.rand(1).item() * 0.01)], dtype=torch.float64)
    mll = ExactMarginalLogLikelihood(model.likelihood, model)
    try:
        fit_gpytorch_mll(mll)
        model.eval()
        with torch.no_grad():
            output = model(X_train)
            loss = -mll(output, model.train_targets).item()
        if abs(loss - best_loss) < abs(best_loss) * 0.1:
            converged_count += 1
    except Exception:
        continue

best_model.eval()

# Extract fitted hyperparameters
ls = best_model.covar_module.base_kernel.lengthscale.detach().squeeze()
noise = best_model.likelihood.noise.detach().item()
outputscale = best_model.covar_module.outputscale.detach().item()

print(f"\n{'='*50}")
print(f"Best MLL loss: {best_loss:.4f}")
print(f"Restarts converged near best (within 10%): {converged_count}/{N_MLL_RESTARTS}")
print(f"\nFitted hyperparameters:")
for i in range(ARD_NUM_DIMS):
    print(f"  Lengthscale dim {i}: {ls[i].item():.6f}")
print(f"  Noise variance:     {noise:.6f} (≥ {NOISE_LB})")
print(f"  Output scale:       {outputscale:.6f}")
assert noise >= NOISE_LB, f"Fitted noise {noise} below floor {NOISE_LB}!"

In [ ]:
# ── Acquisition Optimisation & Candidate Selection ────────────────────────────
# qLogNEI with q=4, distance-based selection to pick best single point
import numpy as np

# Construct acquisition function
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([MC_SAMPLES]))
acqf = qLogNoisyExpectedImprovement(
    model=best_model,
    X_baseline=X_train,
    sampler=sampler,
    prune_baseline=True,
)

# Optimise acquisition over [0,1]⁴
bounds = torch.tensor([[0.0]*ARD_NUM_DIMS, [1.0]*ARD_NUM_DIMS], dtype=torch.float64)
candidates, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=Q_BATCH,
    num_restarts=NUM_RESTARTS,
    raw_samples=RAW_SAMPLES,
)

print(f"=== Acquisition Results (q={Q_BATCH}) ===")
print(f"Acquisition value: {acq_value.item():.6f}\n")

# Get posterior means for each candidate (Standardize auto-untransforms to original space)
best_model.eval()
with torch.no_grad():
    posterior = best_model.posterior(candidates)
    pred_means = posterior.mean.squeeze(-1)  # (Q_BATCH,)

print("All candidates and posterior means (original space):")
for i in range(Q_BATCH):
    c = candidates[i]
    coords = ", ".join(f"{c[j]:.6f}" for j in range(ARD_NUM_DIMS))
    print(f"  Candidate {i+1}: [{coords}]  mean={pred_means[i].item():.6f}")

# Distance-based selection:
# 1. Quality gate: keep candidates with mean >= median
median_mean = pred_means.median().item()
qualified_mask = pred_means >= median_mean
qualified_indices = torch.where(qualified_mask)[0]

print(f"\nMedian posterior mean: {median_mean:.6f}")
print(f"Qualified candidates (mean ≥ median): {len(qualified_indices)}")

# 2. From qualified, pick max min-distance to training data
distances = torch.cdist(candidates[qualified_indices], X_train)
min_distances = distances.min(dim=1).values
best_qualified_idx = min_distances.argmax().item()
selected_idx = qualified_indices[best_qualified_idx].item()

x_new = candidates[selected_idx].detach().numpy()

# Clamp to [0.0, 0.999999]
x_new = np.clip(x_new, 0.0, 0.999999)

# Format submission string
proposed_query = "-".join(f"{v:.6f}" for v in x_new)

# Duplicate check against all existing samples
is_duplicate = False
for i in range(len(inputs)):
    if np.allclose(x_new, inputs[i], atol=1e-6):
        is_duplicate = True
        break

print(f"\n=== Selection ===")
print(f"Selected candidate: {selected_idx + 1} (max min-distance from qualified set)")
print(f"Min distance to training data: {min_distances[best_qualified_idx].item():.6f}")
print(f"Duplicate of existing sample: {is_duplicate}")
print(f"\n>>> SUBMISSION: {proposed_query}")

In [ ]:
# ── 2D Contour Slice Visualisation ────────────────────────────────────────────
# Top-2 most relevant dimensions (shortest ARD lengthscales) — 2 rows (mean + acq)

# Identify top-2 dims by shortest lengthscale
ls_np = ls.detach().cpu().numpy()
sorted_dims = np.argsort(ls_np)
top2 = sorted_dims[:2]
fix_dims = sorted_dims[2:]

dim_a, dim_b = top2[0], top2[1]

print(f"Top-2 dims: x{dim_a}, x{dim_b} (ℓ={ls_np[dim_a]:.4f}, {ls_np[dim_b]:.4f})")
print(f"Fixed dims: " + ", ".join(f"x{d}={x_new[d]:.4f}" for d in fix_dims))

# Build grid over top-2 dims
g0 = np.linspace(0, 1, GRID_RES)
g1 = np.linspace(0, 1, GRID_RES)
G0, G1 = np.meshgrid(g0, g1)

grid_pts = np.tile(x_new, (GRID_RES * GRID_RES, 1))
grid_pts[:, dim_a] = G0.ravel()
grid_pts[:, dim_b] = G1.ravel()
grid_tensor = torch.tensor(grid_pts, dtype=torch.float64)

# GP posterior mean (Standardize auto-untransforms)
best_model.eval()
with torch.no_grad():
    post = best_model.posterior(grid_tensor)
    mean_grid = post.mean.squeeze(-1).cpu().numpy().reshape(GRID_RES, GRID_RES)
    std_grid = post.variance.sqrt().squeeze(-1).cpu().numpy().reshape(GRID_RES, GRID_RES)

# 3-panel: Mean, Std, Dimension relevance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: GP posterior mean
c1 = axes[0].contourf(G0, G1, mean_grid, levels=30, cmap="viridis")
axes[0].scatter(X_train[:N_INITIAL, dim_a].numpy(), X_train[:N_INITIAL, dim_b].numpy(),
                c="tab:blue", edgecolors="white", s=40, zorder=5, label="Initial")
axes[0].scatter(X_train[N_INITIAL:, dim_a].numpy(), X_train[N_INITIAL:, dim_b].numpy(),
                c="tab:orange", edgecolors="white", s=40, zorder=5, label="Submissions")
axes[0].scatter(x_new[dim_a], x_new[dim_b],
                c="lime", s=200, marker="*", edgecolors="black", linewidths=1, label="Proposed", zorder=10)
axes[0].set_xlabel(f"x{dim_a}")
axes[0].set_ylabel(f"x{dim_b}")
axes[0].set_title(f"GP Posterior Mean (x{dim_a} vs x{dim_b})")
axes[0].legend(fontsize=7)
fig.colorbar(c1, ax=axes[0])

# Panel 2: GP posterior std
c2 = axes[1].contourf(G0, G1, std_grid, levels=30, cmap="magma")
axes[1].scatter(X_train[:N_INITIAL, dim_a].numpy(), X_train[:N_INITIAL, dim_b].numpy(),
                c="tab:blue", edgecolors="white", s=40, zorder=5)
axes[1].scatter(X_train[N_INITIAL:, dim_a].numpy(), X_train[N_INITIAL:, dim_b].numpy(),
                c="tab:orange", edgecolors="white", s=40, zorder=5)
axes[1].scatter(x_new[dim_a], x_new[dim_b],
                c="lime", s=200, marker="*", edgecolors="black", linewidths=1, zorder=10)
axes[1].set_xlabel(f"x{dim_a}")
axes[1].set_ylabel(f"x{dim_b}")
axes[1].set_title(f"GP Posterior Std Dev (x{dim_a} vs x{dim_b})")
fig.colorbar(c2, ax=axes[1])

# Panel 3: Dimension relevance
inv_ls = 1.0 / ls_np
inv_ls_norm = inv_ls / inv_ls.sum()
colors = ['darkorange' if d in top2 else 'steelblue' for d in range(ARD_NUM_DIMS)]
axes[2].barh(range(ARD_NUM_DIMS), inv_ls_norm, color=colors)
axes[2].set_yticks(range(ARD_NUM_DIMS))
axes[2].set_yticklabels([f"x{j}" for j in range(ARD_NUM_DIMS)])
axes[2].set_title("Dimension Relevance (1/ℓ, normalised)")
axes[2].set_xlabel("Relative Importance")

fig.suptitle("F4 Week 12 — SFGP Matérn-2.5 ARD Surrogate (Standardize)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Convergence Plot with Proposed Point ──────────────────────────────────────
# Running best in original scale + proposed point from GP prediction

# Predict at proposed point (Standardize auto-untransforms)
x_new_tensor = torch.tensor(x_new, dtype=torch.float64).unsqueeze(0)
best_model.eval()
with torch.no_grad():
    pred_posterior = best_model.posterior(x_new_tensor)
    pred_original = pred_posterior.mean.item()

print(f"=== Proposed Point Prediction ===")
print(f"Predicted value (original scale): {pred_original:.6f}")
print(f"Current best observed: {outputs.max():.6f}")

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 5))

sample_indices = np.arange(1, n_total + 1)

# Running best as line
ax.plot(sample_indices, running_best, "k-", linewidth=1.5, label="Running best")

# Initial samples (blue)
ax.scatter(sample_indices[:N_INITIAL], outputs[:N_INITIAL],
           c="tab:blue", s=30, edgecolors="white", linewidths=0.5, label="Initial samples", zorder=5)

# Submission samples (orange)
ax.scatter(sample_indices[N_INITIAL:], outputs[N_INITIAL:],
           c="tab:orange", s=30, edgecolors="white", linewidths=0.5, label="Submissions", zorder=5)

# Vertical line at initial/submission boundary
ax.axvline(x=N_INITIAL + 0.5, color="gray", linestyle="--", alpha=0.5, label="Initial boundary")

# Proposed point (green star)
ax.scatter(n_total + 1, pred_original,
           c="lime", s=200, marker="*", edgecolors="black", linewidths=1,
           label=f"Proposed ({pred_original:.4f})", zorder=10)

ax.set_xlabel("Sample")
ax.set_ylabel("Output")
ax.set_title("F4 Convergence — Running Best + Proposed (SFGP)")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()